Name: Sufiyan Shaikh

Roll_no:SCFP125021

# Lab 6 — Tools & Tool Calling with LLMs


Sample inventory, order, weather, travel prices, calendar availability, and restaurant details are demonstration data, not live business records. The weather example uses a live API when available.

## 1. Install dependencies

In [1]:
pip install -q -U langchain-core langchain-groq pydantic python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and Groq model setup

In [2]:
import os, json, math, asyncio
from typing import Type, Optional
from pydantic import BaseModel, Field, ConfigDict
from dotenv import load_dotenv
from langchain_core.tools import tool, BaseTool, StructuredTool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_groq import ChatGroq

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
USE_LLM = bool(GROQ_API_KEY)

if USE_LLM:
    # Use a Groq tool-calling capable model.
    llm = ChatGroq(model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"), api_key=GROQ_API_KEY, temperature=0)
    print("Mode: LIVE Groq model")
else:
    llm = None
    print("Mode: OFFLINE DEMO (no GROQ_API_KEY found). Tool functions are real; model tool-call suggestions are simulated.")

def show(title, value):
    print("\n" + "="*12 + " " + title + " " + "="*12)
    if isinstance(value, (dict, list)):
        print(json.dumps(value, indent=2, ensure_ascii=False, default=str))
    else:
        print(value)

def execute_tool_call(tool_obj, args):
    """Execute a LangChain tool with parsed arguments and return a printable result."""
    return tool_obj.invoke(args)

def tool_registry(tools):
    return {t.name: t for t in tools}

def call_tool_by_name(tools, name, args):
    registry = tool_registry(tools)
    if name not in registry:
        return {"error": f"Unknown tool: {name}"}
    try:
        result = registry[name].invoke(args)
        return result
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}

def run_tool_flow(tools, user_query, mock_calls=None, max_rounds=5):
    """Show proposed calls, manually execute them, then ask the model for a final response.
    In offline mode, mock_calls are clearly labelled examples rather than model outputs.
    """
    print("\nUSER:", user_query)
    if not USE_LLM:
        calls = mock_calls or []
        show("OFFLINE DEMO tool-call suggestions (not generated by an LLM)", calls)
        results = []
        for item in calls:
            result = call_tool_by_name(tools, item["name"], item.get("args", {}))
            results.append({"tool": item["name"], "args": item.get("args", {}), "result": result})
        show("Manually executed tool results", results)
        final = "Demo result summary: " + "; ".join(
            f"{r['tool']} returned {r['result']}" for r in results
        ) if results else "No tool call was needed in this offline demonstration."
        show("Final answer (offline demonstration)", final)
        return {"tool_calls": calls, "results": results, "final_answer": final}

    model_with_tools = llm.bind_tools(tools)
    messages = [HumanMessage(content=user_query)]
    all_calls, all_results = [], []
    for _ in range(max_rounds):
        response = model_with_tools.invoke(messages)
        calls = getattr(response, "tool_calls", []) or []
        if not calls:
            show("Model final answer", response.content)
            return {"tool_calls": all_calls, "results": all_results, "final_answer": response.content}
        show("Model suggested tool_calls", calls)
        all_calls.extend(calls)
        messages.append(response)
        for call in calls:
            result = call_tool_by_name(tools, call["name"], call.get("args", {}))
            show("Manually executed " + call["name"], result)
            all_results.append({"tool": call["name"], "args": call.get("args", {}), "result": result})
            messages.append(ToolMessage(
                content=json.dumps(result, ensure_ascii=False, default=str),
                tool_call_id=call["id"]
            ))
    final_response = model_with_tools.invoke(messages)
    show("Model final answer", final_response.content)
    return {"tool_calls": all_calls, "results": all_results, "final_answer": final_response.content}

Mode: LIVE Groq model


## Exercise 1 — Order Status Lookup Tool

In [3]:
@tool
def lookup_order_status(order_id: str) -> str:
    """Look up the shipping status for an order ID in the sample order table."""
    orders = {
        "ORD-4521": "Shipped; expected delivery in 2 days.",
        "ORD-1020": "Processing; not shipped yet.",
        "ORD-7788": "Delivered yesterday."
    }
    return orders.get(order_id.upper(), "Order ID not found in the sample table.")

tools1 = [lookup_order_status]
result1 = run_tool_flow(
    tools1,
    "Where is my order ORD-4521?",
    mock_calls=[{"name":"lookup_order_status","args":{"order_id":"ORD-4521"}}]
)


USER: Where is my order ORD-4521?

============ Model suggested tool_calls ============
[
  {
    "name": "lookup_order_status",
    "args": {
      "order_id": "ORD-4521"
    },
    "id": "fc_16eb4486-6858-404a-a642-51051b96d415",
    "type": "tool_call"
  }
]

============ Manually executed lookup_order_status ============
Shipped; expected delivery in 2 days.

============ Model final answer ============
Your order **ORD‑4521** has already shipped. It’s on its way and should arrive in **2 days**. If you need a tracking number or have any other questions, just let me know!


## Exercise 2 — Loan EMI Calculator Tool

In [4]:
class EMIArgs(BaseModel):
    principal: float = Field(gt=0, description="Loan amount in currency units")
    annual_rate: float = Field(ge=0, le=30, description="Annual interest rate percentage from 0 to 30")
    tenure_months: int = Field(gt=0, description="Loan duration in months")

def calculate_emi(principal: float, annual_rate: float, tenure_months: int) -> str:
    """Calculate monthly loan EMI after validating principal, rate, and tenure."""
    if principal <= 0 or tenure_months <= 0 or not 0 <= annual_rate <= 30:
        raise ValueError("Principal and tenure must be positive; annual rate must be from 0 to 30.")
    r = annual_rate / 1200
    emi = principal / tenure_months if r == 0 else principal * r * (1+r)**tenure_months / ((1+r)**tenure_months - 1)
    return f"Monthly EMI: ₹{emi:,.2f} (principal ₹{principal:,.2f}, annual rate {annual_rate}%, tenure {tenure_months} months)."

emi_tool = StructuredTool.from_function(
    func=calculate_emi, name="calculate_emi",
    description="Calculate monthly loan EMI. Provide principal, annual_rate, and tenure_months.",
    args_schema=EMIArgs
)
tools2 = [emi_tool]
result2 = run_tool_flow(
    tools2, "What's my EMI for a 5 lakh loan at 9% for 24 months?",
    mock_calls=[{"name":"calculate_emi","args":{"principal":500000,"annual_rate":9,"tenure_months":24}}]
)


USER: What's my EMI for a 5 lakh loan at 9% for 24 months?

============ Model suggested tool_calls ============
[
  {
    "name": "calculate_emi",
    "args": {
      "annual_rate": 9,
      "principal": 500000,
      "tenure_months": 24
    },
    "id": "fc_822648ba-dfeb-4119-958c-7d88d45e5aed",
    "type": "tool_call"
  }
]

============ Manually executed calculate_emi ============
Monthly EMI: ₹22,842.37 (principal ₹500,000.00, annual rate 9.0%, tenure 24 months).

============ Model final answer ============
**Monthly EMI:** ₹22,842.37  
*(Principal ₹500,000.00, annual rate 9.0%, tenure 24 months)*


## Exercise 3 — Unit Converter Toolkit (two tools)

In [5]:
@tool
def convert_temperature(value: float, direction: str) -> str:
    """Convert temperature using direction 'C_to_F' or 'F_to_C'."""
    direction = direction.upper()
    if direction == "C_TO_F":
        out = value * 9/5 + 32
        return f"{value:g} °C = {out:.2f} °F"
    if direction == "F_TO_C":
        out = (value - 32) * 5/9
        return f"{value:g} °F = {out:.2f} °C"
    raise ValueError("direction must be C_to_F or F_to_C")

@tool
def convert_distance(value: float, direction: str) -> str:
    """Convert distance using direction 'km_to_miles' or 'miles_to_km'."""
    direction = direction.lower()
    if direction == "km_to_miles":
        return f"{value:g} km = {value * 0.621371:.2f} miles"
    if direction == "miles_to_km":
        return f"{value:g} miles = {value / 0.621371:.2f} km"
    raise ValueError("direction must be km_to_miles or miles_to_km")

tools3 = [convert_temperature, convert_distance]
result3 = run_tool_flow(
    tools3, "Convert 100 km to miles.",
    mock_calls=[{"name":"convert_distance","args":{"value":100,"direction":"km_to_miles"}}]
)


USER: Convert 100 km to miles.

============ Model suggested tool_calls ============
[
  {
    "name": "convert_distance",
    "args": {
      "direction": "km_to_miles",
      "value": 100
    },
    "id": "fc_2a0ab293-2e53-4e17-9b04-7b61db558ebd",
    "type": "tool_call"
  }
]

============ Manually executed convert_distance ============
100 km = 62.14 miles

============ Model final answer ============
100 km ≈ 62.14 miles.


## Exercise 4 — Weather-Based Outfit Recommendation (chained tools)

In [6]:
@tool
def get_weather(city: str) -> str:
    """Return current weather for a city; use Open-Meteo when available, otherwise sample data."""
    import requests
    try:
        g = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                         params={"name":city,"count":1,"language":"en","format":"json"}, timeout=8)
        g.raise_for_status()
        places = g.json().get("results", [])
        if places:
            p = places[0]
            r = requests.get("https://api.open-meteo.com/v1/forecast",
                params={"latitude":p["latitude"],"longitude":p["longitude"],
                        "current":"temperature_2m,weather_code","timezone":"auto"}, timeout=8)
            r.raise_for_status()
            cur = r.json().get("current", {})
            return json.dumps({"city":city,"temperature":cur.get("temperature_2m"),
                               "conditions":f"Weather code {cur.get('weather_code')}","source":"Open-Meteo"})
    except Exception as e:
        print("Weather API fallback:", str(e))
    samples = {"Mumbai": (30, "humid and partly cloudy"), "Pune": (27, "partly cloudy"),
               "Delhi": (29, "hazy")}
    temp, cond = samples.get(city.title(), (25, "mild"))
    return json.dumps({"city":city,"temperature":temp,"conditions":cond,"source":"sample fallback"})

@tool
def recommend_outfit(temperature: float, conditions: str) -> str:
    """Recommend clothing using the provided temperature in Celsius and weather conditions."""
    if temperature >= 32:
        outfit = "light breathable cotton clothes, comfortable footwear, and sun protection"
    elif temperature >= 25:
        outfit = "light clothes; carry a light layer if staying out late"
    elif temperature >= 18:
        outfit = "comfortable clothes with a light jacket"
    else:
        outfit = "warm layers and a jacket"
    if any(x in conditions.lower() for x in ["rain", "storm", "drizzle"]):
        outfit += "; carry an umbrella or raincoat"
    return outfit + f". Weather considered: {temperature}°C, {conditions}."

tools4 = [get_weather, recommend_outfit]
result4 = run_tool_flow(
    tools4, "What should I wear today in Mumbai?",
    mock_calls=[
        {"name":"get_weather","args":{"city":"Mumbai"}},
        {"name":"recommend_outfit","args":{"temperature":30,"conditions":"humid and partly cloudy"}}
    ]
)


USER: What should I wear today in Mumbai?

============ Model suggested tool_calls ============
[
  {
    "name": "get_weather",
    "args": {
      "city": "Mumbai"
    },
    "id": "fc_7e0dd2db-83f3-47e1-aaaf-95c2632854dc",
    "type": "tool_call"
  }
]

============ Manually executed get_weather ============
{"city": "Mumbai", "temperature": 31.1, "conditions": "Weather code 1", "source": "Open-Meteo"}

============ Model suggested tool_calls ============
[
  {
    "name": "recommend_outfit",
    "args": {
      "conditions": "Weather code 1",
      "temperature": 31.1
    },
    "id": "fc_d18630c6-2c29-4b63-8323-b84d102980e3",
    "type": "tool_call"
  }
]

============ Manually executed recommend_outfit ============
light clothes; carry a light layer if staying out late. Weather considered: 31.1°C, Weather code 1.

============ Model final answer ============
**Mumbai today – 31.1 °C, clear skies (weather code 1)**  

- **Top:** Light, breathable cotton or linen T‑shirt or blouse

## Exercise 5 — Async Product Stock Checker (`BaseTool`)

In [7]:
class ProductStockTool(BaseTool):
    name: str = "check_product_stock"
    description: str = "Check whether a product is in stock by product_id."
    def _run(self, product_id: str) -> str:
        stock = {"P1023": True, "P2044": False, "P3099": True}
        if product_id.upper() not in stock:
            return f"Product {product_id}: not found in sample inventory."
        return f"Product {product_id}: " + ("in stock." if stock[product_id.upper()] else "out of stock.")
    async def _arun(self, product_id: str) -> str:
        # Async implementation; no blocking network calls are used.
        return self._run(product_id)

stock_tool = ProductStockTool()
print("Manual sync .invoke():", stock_tool.invoke({"product_id":"P1023"}))

async def check_async():
    return await stock_tool.ainvoke({"product_id":"P1023"})

print("Manual async .ainvoke():", await check_async())

result5 = run_tool_flow(
    [stock_tool], "Is product P1023 in stock?",
    mock_calls=[{"name":"check_product_stock","args":{"product_id":"P1023"}}]
)

Manual sync .invoke(): Product P1023: in stock.
Manual async .ainvoke(): Product P1023: in stock.

USER: Is product P1023 in stock?

============ Model suggested tool_calls ============
[
  {
    "name": "check_product_stock",
    "args": {
      "product_id": "P1023"
    },
    "id": "fc_3b314196-bb3c-44b7-9c22-d73f233a28cf",
    "type": "tool_call"
  }
]

============ Manually executed check_product_stock ============
Product P1023: in stock.

============ Model final answer ============
Yes—Product **P1023** is currently in stock.


## Exercise 6 — Expense Splitter Tool

In [8]:
class ExpenseArgs(BaseModel):
    total_amount: float = Field(gt=0, description="Total bill amount")
    num_people: int = Field(gt=0, description="Number of people sharing the bill")

def split_expense(total_amount: float, num_people: int) -> str:
    """Split a positive total bill equally among a positive number of people."""
    if total_amount <= 0 or num_people <= 0:
        raise ValueError("total_amount and num_people must be positive.")
    return f"Each person owes ₹{total_amount / num_people:.2f}."

expense_tool = StructuredTool.from_function(
    func=split_expense, name="split_expense",
    description="Split a bill equally among people.",
    args_schema=ExpenseArgs
)
manual6 = expense_tool.invoke({"total_amount":2400,"num_people":4})
show("Manual result", manual6)
result6 = run_tool_flow(
    [expense_tool], "Split ₹2400 between 4 friends.",
    mock_calls=[{"name":"split_expense","args":{"total_amount":2400,"num_people":4}}]
)
print("Manual result matches bound-flow result:",
      manual6 == result6["results"][0]["result"] if result6["results"] else "Compare after live execution.")


============ Manual result ============
Each person owes ₹600.00.

USER: Split ₹2400 between 4 friends.

============ Model suggested tool_calls ============
[
  {
    "name": "split_expense",
    "args": {
      "num_people": 4,
      "total_amount": 2400
    },
    "id": "fc_e36bba65-bbf6-48e5-bda5-c2f10622380e",
    "type": "tool_call"
  }
]

============ Manually executed split_expense ============
Each person owes ₹600.00.

============ Model final answer ============
Each person owes ₹600.00.
Manual result matches bound-flow result: True


## Exercise 7 — Trip Cost Estimator (three independent tools)

In [10]:
@tool
def get_flight_cost(origin: str, destination: str) -> str:
    """Return a sample estimated round-trip flight cost between two cities in INR."""
    fares = {("Delhi","Goa"):8500, ("Pune","Delhi"):6000}
    cost = fares.get((origin.title(), destination.title()), 9000)
    return f"Estimated round-trip flight cost from {origin} to {destination}: ₹{cost:,} (sample estimate)."

@tool
def get_hotel_cost(city: str, nights: int) -> str:
    """Return sample hotel cost in INR for the requested number of nights."""
    if nights <= 0: raise ValueError("nights must be positive.")
    nightly = {"Goa":3500, "Delhi":3000, "Pune":2800}.get(city.title(), 3000)
    return f"Estimated hotel cost in {city} for {nights} nights: ₹{nightly*nights:,} (sample estimate)."

@tool
def get_food_budget(city: str, days: int) -> str:
    """Return sample food budget in INR for the requested number of days."""
    if days <= 0: raise ValueError("days must be positive.")
    daily = {"Goa":900, "Delhi":1000, "Pune":800}.get(city.title(), 900)
    return f"Estimated food budget in {city} for {days} days: ₹{daily*days:,} (sample estimate)."

tools7 = [get_flight_cost, get_hotel_cost, get_food_budget]
result7 = run_tool_flow(
    tools7, "I'm planning a 3-night trip from Delhi to Goa. Estimate flights, hotel, and food.",
    mock_calls=[
        {"name":"get_flight_cost","args":{"origin":"Delhi","destination":"Goa"}},
        {"name":"get_hotel_cost","args":{"city":"Goa","nights":3}},
        {"name":"get_food_budget","args":{"city":"Goa","days":4}}
    ]
)


USER: I'm planning a 3-night trip from Delhi to Goa. Estimate flights, hotel, and food.

============ Model suggested tool_calls ============
[
  {
    "name": "get_flight_cost",
    "args": {
      "destination": "Goa",
      "origin": "Delhi"
    },
    "id": "fc_761ece04-9802-40f8-bf93-0638493a2eec",
    "type": "tool_call"
  }
]

============ Manually executed get_flight_cost ============
Estimated round-trip flight cost from Delhi to Goa: ₹8,500 (sample estimate).

============ Model suggested tool_calls ============
[
  {
    "name": "get_hotel_cost",
    "args": {
      "city": "Goa",
      "nights": 3
    },
    "id": "fc_7dcca188-4af5-4f69-b1fe-d7269616610a",
    "type": "tool_call"
  }
]

============ Manually executed get_hotel_cost ============
Estimated hotel cost in Goa for 3 nights: ₹10,500 (sample estimate).

============ Model suggested tool_calls ============
[
  {
    "name": "get_food_budget",
    "args": {
      "city": "Goa",
      "days": 3
    },
    "id": "fc_

## Exercise 8 — Health Metrics Toolkit (BMI + calorie needs)

In [11]:
class BMIArgs(BaseModel):
    weight_kg: float = Field(gt=0)
    height_cm: float = Field(gt=0)

class CalorieArgs(BaseModel):
    weight_kg: float = Field(gt=0)
    height_cm: float = Field(gt=0)
    age: int = Field(gt=0)
    activity_level: str = Field(description="sedentary, lightly_active, moderately_active, very_active")

def calculate_bmi(weight_kg: float, height_cm: float) -> str:
    """Calculate BMI from weight in kg and height in cm."""
    bmi = weight_kg / (height_cm / 100) ** 2
    return f"BMI: {bmi:.1f}. Adult BMI categories are only a rough screening measure."

def calculate_daily_calories(weight_kg: float, height_cm: float, age: int, activity_level: str) -> str:
    """Estimate daily calories using a simplified Mifflin-St Jeor formula with a neutral baseline; this estimate is approximate."""
    factors = {"sedentary":1.2, "lightly_active":1.375, "moderately_active":1.55, "very_active":1.725}
    factor = factors.get(activity_level.lower())
    if factor is None:
        raise ValueError("activity_level must be sedentary, lightly_active, moderately_active, or very_active.")
    # Sex-specific formula is not available from the lab inputs, so use an explicitly approximate neutral estimate.
    bmr = 10*weight_kg + 6.25*height_cm - 5*age - 78
    return f"Approximate daily calorie maintenance estimate: {bmr*factor:.0f} kcal/day (formula estimate, not medical advice)."

bmi_tool = StructuredTool.from_function(func=calculate_bmi, name="calculate_bmi",
    description="Calculate BMI from weight_kg and height_cm.", args_schema=BMIArgs)
calorie_tool = StructuredTool.from_function(func=calculate_daily_calories, name="calculate_daily_calories",
    description="Estimate daily calorie needs from weight, height, age, and activity level.", args_schema=CalorieArgs)

class HealthToolkit:
    def get_tools(self):
        return [bmi_tool, calorie_tool]

health_tools = HealthToolkit().get_tools()
show("Toolkit tools", [t.name for t in health_tools])
result8 = run_tool_flow(
    health_tools,
    "I'm 70kg, 175cm tall, 25 years old, moderately active. What's my BMI and daily calorie need?",
    mock_calls=[
        {"name":"calculate_bmi","args":{"weight_kg":70,"height_cm":175}},
        {"name":"calculate_daily_calories","args":{"weight_kg":70,"height_cm":175,"age":25,"activity_level":"moderately_active"}}
    ]
)


============ Toolkit tools ============
[
  "calculate_bmi",
  "calculate_daily_calories"
]

USER: I'm 70kg, 175cm tall, 25 years old, moderately active. What's my BMI and daily calorie need?

============ Model suggested tool_calls ============
[
  {
    "name": "calculate_bmi",
    "args": {
      "height_cm": 175,
      "weight_kg": 70
    },
    "id": "fc_2073e776-d555-44bf-a07e-66a605a00b2b",
    "type": "tool_call"
  }
]

============ Manually executed calculate_bmi ============
BMI: 22.9. Adult BMI categories are only a rough screening measure.

============ Model suggested tool_calls ============
[
  {
    "name": "calculate_daily_calories",
    "args": {
      "activity_level": "moderately_active",
      "age": 25,
      "height_cm": 175,
      "weight_kg": 70
    },
    "id": "fc_12d6f243-47bb-4c65-ba8d-a7300a3248b6",
    "type": "tool_call"
  }
]

============ Manually executed calculate_daily_calories ============
Approximate daily calorie maintenance estimate: 2466 kcal/d

## Exercise 9 — Smart Meeting Scheduler (partial tool use)

In [12]:
@tool
def check_calendar_availability(person: str, date: str) -> str:
    """Check sample calendar availability for a person on YYYY-MM-DD."""
    return f"{person} is available on {date} between 10:00 and 12:00 (sample calendar)."

@tool
def get_timezone_offset(city: str) -> str:
    """Return a sample timezone offset for a city."""
    zones = {"Pune":"+05:30", "Mumbai":"+05:30", "London":"+00:00", "New York":"-04:00"}
    return f"{city}: UTC{zones.get(city.title(), '+00:00')} (sample timezone data)."

@tool
def book_meeting(person: str, date: str, time: str) -> str:
    """Create a sample meeting booking confirmation."""
    return f"Meeting booked with {person} for {date} at {time} (sample booking; no real calendar was changed)."

tools9 = [check_calendar_availability, get_timezone_offset, book_meeting]
availability9 = run_tool_flow(
    tools9, "Check whether Asha is available on 2026-10-26.",
    mock_calls=[{"name":"check_calendar_availability","args":{"person":"Asha","date":"2026-10-26"}}]
)
booking9 = run_tool_flow(
    tools9, "Schedule a meeting with Asha on 2026-10-26 at 10:30 Pune time. Check availability and timezone first.",
    mock_calls=[
        {"name":"check_calendar_availability","args":{"person":"Asha","date":"2026-10-26"}},
        {"name":"get_timezone_offset","args":{"city":"Pune"}},
        {"name":"book_meeting","args":{"person":"Asha","date":"2026-10-26","time":"10:30 UTC+05:30"}}
    ]
)
show("Availability query tool names", [c["name"] for c in availability9["tool_calls"]])
show("Booking query tool names", [c["name"] for c in booking9["tool_calls"]])


USER: Check whether Asha is available on 2026-10-26.

============ Model suggested tool_calls ============
[
  {
    "name": "check_calendar_availability",
    "args": {
      "date": "2026-10-26",
      "person": "Asha"
    },
    "id": "fc_f232de24-5dd9-4562-ac75-a6027898792f",
    "type": "tool_call"
  }
]

============ Manually executed check_calendar_availability ============
Asha is available on 2026-10-26 between 10:00 and 12:00 (sample calendar).

============ Model final answer ============
Asha is available on **2026‑10‑26** between **10:00 am and 12:00 pm** (sample calendar).

USER: Schedule a meeting with Asha on 2026-10-26 at 10:30 Pune time. Check availability and timezone first.

============ Model suggested tool_calls ============
[
  {
    "name": "check_calendar_availability",
    "args": {
      "date": "2026-10-26",
      "person": "Asha"
    },
    "id": "fc_cc4b2f95-c0b9-45b5-afa6-db9272ac58b1",
    "type": "tool_call"
  }
]

============ Manually executed check_

## Exercise 10 — Restaurant Recommendation Assistant (four tools)

In [13]:
@tool
def find_restaurants(cuisine: str, city: str) -> str:
    """Find sample restaurants matching cuisine and city."""
    sample = {
        ("italian","pune"): ["Bella Italia", "Little Italy"],
        ("indian","pune"): ["Shabree", "Durvankur"]
    }
    return json.dumps(sample.get((cuisine.lower(), city.lower()), [f"Sample {cuisine.title()} Restaurant A", f"Sample {cuisine.title()} Restaurant B"]))

@tool
def get_restaurant_rating(restaurant_name: str) -> str:
    """Return a sample customer rating out of five."""
    ratings = {"Bella Italia":4.3, "Little Italy":4.1, "Shabree":4.4, "Durvankur":4.2}
    return f"{restaurant_name}: {ratings.get(restaurant_name, 4.0):.1f}/5 (sample rating)."

@tool
def check_restaurant_open(restaurant_name: str, day: str) -> str:
    """Check sample opening status for a restaurant and day."""
    return f"{restaurant_name} is listed as open on {day} in this sample dataset."

@tool
def estimate_cost_for_two(restaurant_name: str) -> str:
    """Estimate sample meal cost for two people in INR."""
    prices = {"Bella Italia":1800, "Little Italy":1600, "Shabree":900, "Durvankur":700}
    return f"Estimated cost for two at {restaurant_name}: ₹{prices.get(restaurant_name, 1200):,} (sample estimate)."

tools10 = [find_restaurants, get_restaurant_rating, check_restaurant_open, estimate_cost_for_two]
result10 = run_tool_flow(
    tools10, "Suggest a good Italian restaurant in Pune open today, with rating and rough cost for two.",
    mock_calls=[
        {"name":"find_restaurants","args":{"cuisine":"Italian","city":"Pune"}},
        {"name":"get_restaurant_rating","args":{"restaurant_name":"Bella Italia"}},
        {"name":"check_restaurant_open","args":{"restaurant_name":"Bella Italia","day":"today"}},
        {"name":"estimate_cost_for_two","args":{"restaurant_name":"Bella Italia"}}
    ]
)


USER: Suggest a good Italian restaurant in Pune open today, with rating and rough cost for two.

============ Model suggested tool_calls ============
[
  {
    "name": "find_restaurants",
    "args": {
      "city": "Pune",
      "cuisine": "Italian"
    },
    "id": "fc_46074d7e-2068-4b1c-9d6f-48d2a2ff57bb",
    "type": "tool_call"
  }
]

============ Manually executed find_restaurants ============
["Bella Italia", "Little Italy"]

============ Model suggested tool_calls ============
[
  {
    "name": "check_restaurant_open",
    "args": {
      "day": "Monday",
      "restaurant_name": "Bella Italia"
    },
    "id": "fc_2720f3e4-22a6-4b8e-81e5-566fecdb851e",
    "type": "tool_call"
  }
]

============ Manually executed check_restaurant_open ============
Bella Italia is listed as open on Monday in this sample dataset.

============ Model suggested tool_calls ============
[
  {
    "name": "get_restaurant_rating",
    "args": {
      "restaurant_name": "Bella Italia"
    },
    "id":